# SHL Grammar Scoring Engine

Multimodal regression pipeline for predicting grammar scores from spoken English audio.

**Final approach:** Whisper-small transcript features + Wav2Vec2 speech representations + handcrafted acoustic features + CatBoost regression.

## 1. Setup and data

The competition data is accessed directly from Kaggle's mounted input directory. No competition data is stored in this repository.

In [ ]:
!pip -q install transformers soundfile faster-whisper catboost

import os
import re
import numpy as np
import pandas as pd
import torch
import librosa

from tqdm.auto import tqdm
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr

BASE_DIR = "/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final"
TRAIN_DIR = os.path.join(BASE_DIR, "train")
TEST_DIR = os.path.join(BASE_DIR, "test")

train_df = pd.read_csv(os.path.join(BASE_DIR, "train.csv"))
test_df = pd.read_csv(os.path.join(BASE_DIR, "test.csv"))

y = train_df["label"].values.astype(np.float32)

print("Train:", train_df.shape)
print("Test:", test_df.shape)
print("GPU:", torch.cuda.is_available())

## 2. Whisper-small transcription

Whisper-small is used only to obtain transcripts. Four compact lexical/syntactic surface features were selected for the final model based on validation experiments.

In [ ]:
from faster_whisper import WhisperModel

fw_model = WhisperModel(
    "small",
    device="cuda" if torch.cuda.is_available() else "cpu",
    compute_type="float16" if torch.cuda.is_available() else "int8"
)

def transcribe_files(audio_dir, filenames):
    texts = []
    segment_counts = []

    for filename in tqdm(filenames, desc="Whisper transcription"):
        path = os.path.join(audio_dir, filename)

        audio, _ = librosa.load(
            path,
            sr=16000,
            mono=True
        )

        segments, _ = fw_model.transcribe(
            audio,
            language="en",
            beam_size=5,
            vad_filter=True
        )

        segments = list(segments)

        texts.append(
            " ".join(s.text.strip() for s in segments).strip()
        )
        segment_counts.append(len(segments))

    return pd.DataFrame({
        "filename": filenames,
        "text": texts,
        "segment_count": segment_counts
    })

train_asr_df = transcribe_files(
    TRAIN_DIR,
    train_df["filename"].tolist()
)

test_asr_df = transcribe_files(
    TEST_DIR,
    test_df["filename"].tolist()
)

print("Train transcripts:", train_asr_df.shape)
print("Test transcripts:", test_asr_df.shape)
print("Empty train transcripts:",
      (train_asr_df["text"].str.strip() == "").sum())
print("Empty test transcripts:",
      (test_asr_df["text"].str.strip() == "").sum())

In [ ]:

FILLERS = {
    "um", "uh", "erm", "hmm", "like", "actually", "basically",
    "you know", "i mean", "well"
}

def extract_text_features(text):
    text = str(text).strip()
    
    if not text:
        return {
            "word_count": 0,
            "unique_word_ratio": 0,
            "sentence_count": 0,
            "avg_sentence_length": 0,
            "sentence_length_std": 0,
            "repeated_word_ratio": 0,
            "filler_ratio": 0,
            "question_ratio": 0,
            "exclamation_ratio": 0,
            "chars_per_word": 0,
        }
    
    words = re.findall(r"\b[a-zA-Z']+\b", text.lower())
    word_count = len(words)
    
    unique_word_ratio = len(set(words)) / max(word_count, 1)
    
    sentences = [s.strip() for s in re.split(r"[.!?]+", text) if s.strip()]
    sentence_count = len(sentences)
    
    sent_lengths = [
        len(re.findall(r"\b[a-zA-Z']+\b", s))
        for s in sentences
    ]
    
    avg_sentence_length = (
        np.mean(sent_lengths) if sent_lengths else 0
    )
    sentence_length_std = (
        np.std(sent_lengths) if len(sent_lengths) > 1 else 0
    )
    
    repeated_word_ratio = (
        1 - unique_word_ratio if word_count > 0 else 0
    )
    
    filler_count = 0
    for filler in FILLERS:
        if " " in filler:
            filler_count += len(re.findall(
                r"\b" + re.escape(filler) + r"\b", text.lower()
            ))
        else:
            filler_count += words.count(filler)
    
    filler_ratio = filler_count / max(word_count, 1)
    
    question_ratio = text.count("?") / max(sentence_count, 1)
    exclamation_ratio = text.count("!") / max(sentence_count, 1)
    
    chars_per_word = len(text) / max(word_count, 1)
    
    return {
        "word_count": word_count,
        "unique_word_ratio": unique_word_ratio,
        "sentence_count": sentence_count,
        "avg_sentence_length": avg_sentence_length,
        "sentence_length_std": sentence_length_std,
        "repeated_word_ratio": repeated_word_ratio,
        "filler_ratio": filler_ratio,
        "question_ratio": question_ratio,
        "exclamation_ratio": exclamation_ratio,
        "chars_per_word": chars_per_word,
    }

train_text_features = pd.DataFrame(
    [extract_text_features(t) for t in train_asr_df["text"]]
)

test_text_features = pd.DataFrame(
    [extract_text_features(t) for t in test_asr_df["text"]]
)

print("Train:", train_text_features.shape)
print("Test:", test_text_features.shape)
print("Train text features:", train_text_features.shape)
print("Test text features:", test_text_features.shape)

## 3. Wav2Vec2 speech representations

A pretrained `facebook/wav2vec2-base-960h` encoder is used as a fixed speech representation model. Mean and standard-deviation pooling over the hidden states produces a 1,536-dimensional representation per recording.

In [ ]:
from transformers import Wav2Vec2Processor, Wav2Vec2Model

W2V_MODEL_NAME = "facebook/wav2vec2-base-960h"
device = "cuda" if torch.cuda.is_available() else "cpu"

processor = Wav2Vec2Processor.from_pretrained(W2V_MODEL_NAME)
w2v_model = Wav2Vec2Model.from_pretrained(W2V_MODEL_NAME).to(device)
w2v_model.eval()

print("Wav2Vec2 loaded on:", device)

In [ ]:
import os
import numpy as np
import torch
import librosa
from tqdm.auto import tqdm

def extract_w2v_embeddings(audio_dir, filenames, batch_size=8):
    all_embeddings = []

    for i in tqdm(range(0, len(filenames), batch_size), desc="Wav2Vec2"):
        batch_files = filenames[i:i+batch_size]
        audios = []

        for fname in batch_files:
            path = os.path.join(audio_dir, fname)
            audio, _ = librosa.load(path, sr=16000, mono=True)
            audios.append(audio)

        inputs = processor(
            audios,
            sampling_rate=16000,
            return_tensors="pt",
            padding=True
        )

        input_values = inputs.input_values.to("cuda")

        with torch.no_grad():
            hidden = w2v_model(
                input_values=input_values
            ).last_hidden_state

        # Processor doesn't provide attention_mask in this environment.
        # Approximate valid hidden length from each original audio length.
        input_lengths = torch.tensor(
            [len(a) for a in audios],
            device="cuda"
        )

        # Wav2Vec2 downsamples the waveform by approximately 320x.
        hidden_lengths = torch.ceil(input_lengths / 320).long()
        hidden_lengths = torch.clamp(hidden_lengths, max=hidden.shape[1])

        pooled = []

        for j, length in enumerate(hidden_lengths):
            h = hidden[j, :length]
            mean = h.mean(dim=0)
            std = h.std(dim=0)
            pooled.append(torch.cat([mean, std]))

        embeddings = torch.stack(pooled)
        all_embeddings.append(
            embeddings.cpu().numpy().astype(np.float32)
        )

    return np.vstack(all_embeddings)

train_files = train_df["filename"].tolist()
test_files = test_df["filename"].tolist()

X_w2v_train = extract_w2v_embeddings(TRAIN_DIR, train_files)
X_w2v_test = extract_w2v_embeddings(TEST_DIR, test_files)

print("Train:", X_w2v_train.shape)
print("Test:", X_w2v_test.shape)

np.save("/kaggle/working/X_w2v_train.npy", X_w2v_train)
np.save("/kaggle/working/X_w2v_test.npy", X_w2v_test)

print("Saved.")

## 4. Handcrafted acoustic features

The acoustic feature set captures duration, energy, spectral characteristics, zero-crossing behaviour, speech activity, silence, and pause statistics.

In [ ]:
import os
import numpy as np
import librosa
from tqdm.auto import tqdm

def extract_acoustic_features(audio_dir, filenames):
    features = []

    for fname in tqdm(filenames, desc="Acoustic features"):
        path = os.path.join(audio_dir, fname)
        y_audio, sr = librosa.load(path, sr=16000, mono=True)

        duration = len(y_audio) / sr

        rms = librosa.feature.rms(y=y_audio)[0]
        zcr = librosa.feature.zero_crossing_rate(y_audio)[0]
        centroid = librosa.feature.spectral_centroid(y=y_audio, sr=sr)[0]
        bandwidth = librosa.feature.spectral_bandwidth(y=y_audio, sr=sr)[0]
        rolloff = librosa.feature.spectral_rolloff(y=y_audio, sr=sr)[0]

        # Energy-based speech/activity estimate
        threshold = 0.02 * (np.max(np.abs(y_audio)) + 1e-8)
        active = np.abs(y_audio) > threshold
        active_ratio = active.mean()
        active_duration = active_ratio * duration

        # Silence ratio
        silence_ratio = 1.0 - active_ratio

        # Pause statistics from contiguous inactive regions
        inactive = ~active
        changes = np.diff(np.concatenate([[0], inactive.astype(int), [0]]))
        starts = np.where(changes == 1)[0]
        ends = np.where(changes == -1)[0]

        pauses = (ends - starts) / sr
        pauses = pauses[pauses > 0.15]

        if len(pauses) == 0:
            pause_count = 0
            pause_mean = 0
            pause_std = 0
            pause_max = 0
            pause_total = 0
        else:
            pause_count = len(pauses)
            pause_mean = pauses.mean()
            pause_std = pauses.std()
            pause_max = pauses.max()
            pause_total = pauses.sum()

        features.append([
            duration,
            rms.mean(),
            rms.std(),
            rms.max(),
            zcr.mean(),
            zcr.std(),
            centroid.mean(),
            centroid.std(),
            bandwidth.mean(),
            rolloff.mean(),
            silence_ratio,
            active_ratio,
            active_duration,
            pause_count,
            pause_mean,
            pause_std,
            pause_max,
            pause_total,
        ])

    return np.asarray(features, dtype=np.float32)

X_acoustic_train = extract_acoustic_features(TRAIN_DIR, train_files)
X_acoustic_test = extract_acoustic_features(TEST_DIR, test_files)

print("Train:", X_acoustic_train.shape)
print("Test:", X_acoustic_test.shape)
print("NaNs:", np.isnan(X_acoustic_train).sum(), np.isnan(X_acoustic_test).sum())

np.save("/kaggle/working/X_acoustic_train.npy", X_acoustic_train)
np.save("/kaggle/working/X_acoustic_test.npy", X_acoustic_test)

## 5. Final feature matrix

The final model combines 1,536 Wav2Vec2 features, 18 acoustic features, and four selected transcript features:

- `word_count`
- `unique_word_ratio`
- `chars_per_word`
- `sentence_length_std`

In [ ]:
target_text_cols = [
    "word_count",
    "unique_word_ratio",
    "chars_per_word",
    "sentence_length_std",
]

X_final_train = np.hstack([
    X_w2v_train,
    X_acoustic_train,
    train_text_features[target_text_cols].values.astype(np.float32)
])

X_final_test = np.hstack([
    X_w2v_test,
    X_acoustic_test,
    test_text_features[target_text_cols].values.astype(np.float32)
])

print("Final train shape:", X_final_train.shape)
print("Final test shape:", X_final_test.shape)

## 6. Five-fold cross-validation

Model selection used shuffled 5-fold cross-validation with RMSE and Pearson correlation.

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_pred = np.zeros(len(y), dtype=np.float32)

for fold, (tr_idx, va_idx) in enumerate(kf.split(X_final_train), 1):
    model = CatBoostRegressor(
        iterations=700,
        depth=6,
        learning_rate=0.03,
        l2_leaf_reg=5,
        loss_function="RMSE",
        random_seed=42,
        verbose=False,
        allow_writing_files=False,
        task_type="GPU" if torch.cuda.is_available() else "CPU"
    )

    model.fit(X_final_train[tr_idx], y[tr_idx])
    oof_pred[va_idx] = model.predict(X_final_train[va_idx])

    fold_rmse = mean_squared_error(
        y[va_idx],
        oof_pred[va_idx]
    ) ** 0.5

    print(f"Fold {fold}: RMSE {fold_rmse:.4f}")

cv_rmse = mean_squared_error(y, oof_pred) ** 0.5
cv_pearson = pearsonr(y, oof_pred)[0]

print("\nCross-validation result")
print("RMSE:", round(cv_rmse, 4))
print("Pearson:", round(cv_pearson, 4))

## 7. Final model and submission

The selected CatBoost configuration is trained on all available training examples. The required in-sample training RMSE is reported, and predictions are clipped to the valid 0–5 grammar-score range.

In [ ]:
final_model = CatBoostRegressor(
    iterations=700,
    depth=6,
    learning_rate=0.03,
    l2_leaf_reg=5,
    loss_function="RMSE",
    random_seed=42,
    verbose=False,
    allow_writing_files=False,
    task_type="GPU" if torch.cuda.is_available() else "CPU"
)

final_model.fit(X_final_train, y)

# Required training RMSE
train_pred = final_model.predict(X_final_train)
train_rmse = mean_squared_error(y, train_pred) ** 0.5

# Test predictions
test_pred = final_model.predict(X_final_test)
test_pred = np.clip(test_pred, 0, 5)

print("FINAL TRAINING RMSE:", round(train_rmse, 4))
print("Test mean:", round(test_pred.mean(), 4))
print("Test std:", round(test_pred.std(), 4))
print("Test min:", round(test_pred.min(), 4))
print("Test max:", round(test_pred.max(), 4))
print("NaNs:", np.isnan(test_pred).sum())

submission = pd.DataFrame({
    "filename": test_df["filename"],
    "label": test_pred
})

submission_path = "/kaggle/working/submission_final.csv"
submission.to_csv(submission_path, index=False)

print("Saved:", submission_path)
display(submission.head())

## 8. Final result

The submitted model achieved a **0.5073 Kaggle leaderboard score**.

The local five-fold CV result for the final feature/model configuration was **RMSE 0.6608** and **Pearson 0.8466**.